# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anujrkt06-tech/Flyrank-ML-project-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

I use February information as the feature window. I use GSC performance, position, AI sessions, and GA4 engagement information that was available before the prediction window. I create CTR and average-position features. Numeric missing values are filled with 0, and GA4 availability is kept as a separate categorical feature.

In [17]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(COALESCE(gsc_impressions, 0)) AS imp_feb,
    SUM(COALESCE(gsc_clicks, 0)) AS clk_feb,

    SUM(COALESCE(gsc_sum_position, 0)) AS pos_sum_feb,

    SUM(
        CASE WHEN gsc_data_available = TRUE
             THEN COALESCE(sessions_ai, 0)
             ELSE 0 END
    ) AS sessions_ai_feb,

    SUM(
        CASE WHEN ga4_data_available = TRUE
             THEN COALESCE(ga4_pageviews, 0)
             ELSE 0 END
    ) AS ga4_pageviews_feb,

    SUM(
        CASE WHEN ga4_data_available = TRUE
             THEN COALESCE(ga4_sessions, 0)
             ELSE 0 END
    ) AS ga4_sessions_feb,

    SUM(
        CASE WHEN ga4_data_available = TRUE
             THEN COALESCE(ga4_total_engagement_sec, 0)
             ELSE 0 END
    ) AS ga4_engagement_sec_feb,

    MAX(CASE WHEN ga4_data_available = TRUE THEN 1 ELSE 0 END)
        AS ga4_available_feb

FROM {FEB}
GROUP BY client_hash_id, content_hash_id
""").df()

# Engineered features
features["ctr_feb"] = np.where(
    features["imp_feb"] > 0,
    features["clk_feb"] / features["imp_feb"],
    0
)

features["avg_position_feb"] = np.where(
    features["imp_feb"] > 0,
    features["pos_sum_feb"] / features["imp_feb"],
    0
)

# Remove intermediate feature
features = features.drop(columns=["pos_sum_feb"])

# Build model-ready vector
X = features.drop(
    columns=["client_hash_id", "content_hash_id"]
).copy()

# Fill numeric missing values
X = X.fillna(0)

print("Feature vector shape:", X.shape)
print(X.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (321546, 9)
   imp_feb  clk_feb  sessions_ai_feb  ga4_pageviews_feb  ga4_sessions_feb  \
0      0.0      0.0              0.0                0.0               0.0   
1      0.0      0.0              0.0                0.0               0.0   
2      0.0      0.0              0.0                0.0               0.0   
3      0.0      0.0              0.0                0.0               0.0   
4      0.0      0.0              0.0                0.0               0.0   

   ga4_engagement_sec_feb  ga4_available_feb  ctr_feb  avg_position_feb  
0                     0.0                  0      0.0               0.0  
1                     0.0                  0      0.0               0.0  
2                     0.0                  0      0.0               0.0  
3                     0.0                  0      0.0               0.0  
4                     0.0                  0      0.0               0.0  


## 2. Feature notes (meaning, missing, categorical, available-when?)

imp_feb: February impressions. Missing values are filled with 0. Available before prediction.

clk_feb: February clicks. Missing values are filled with 0. Available before prediction.

ctr_feb: February click-through rate, calculated as clicks divided by impressions. Zero-impression cases become 0. Available before prediction.

avg_position_feb: February average search position, calculated from position sum divided by impressions. Missing or zero-impression cases become 0. Available before prediction.

sessions_ai_feb: February AI sessions. Missing values are filled with 0. Available before prediction.

ga4_pageviews_feb: February GA4 pageviews where GA4 data was available. Missing values are filled with 0.

ga4_sessions_feb: February GA4 sessions where GA4 data was available. Missing values are filled with 0.

ga4_engagement_sec_feb: February GA4 engagement seconds where GA4 data was available. Missing values are filled with 0.

ga4_available_feb: categorical availability indicator for GA4 information. It is represented as 0/1. Available before prediction.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Missing values:")
print(X.isna().sum())

print("\nFeature names:")
print(X.columns.tolist())

print("\nFeature types:")
print(X.dtypes)

Missing values:
imp_feb                   0
clk_feb                   0
sessions_ai_feb           0
ga4_pageviews_feb         0
ga4_sessions_feb          0
ga4_engagement_sec_feb    0
ga4_available_feb         0
ctr_feb                   0
avg_position_feb          0
dtype: int64

Feature names:
['imp_feb', 'clk_feb', 'sessions_ai_feb', 'ga4_pageviews_feb', 'ga4_sessions_feb', 'ga4_engagement_sec_feb', 'ga4_available_feb', 'ctr_feb', 'avg_position_feb']

Feature types:
imp_feb                   float64
clk_feb                   float64
sessions_ai_feb           float64
ga4_pageviews_feb         float64
ga4_sessions_feb          float64
ga4_engagement_sec_feb    float64
ga4_available_feb           int32
ctr_feb                   float64
avg_position_feb          float64
dtype: object


## 3. The leakage hunt

I checked the feature names for label-derived fields and future-window information. March clicks, March impressions, the future label, outcome fields, and other future information are not used in X. I also exclude product/client capability flags because they are context rather than predictive performance features.

In [19]:
# This cell is for CODE
# March is the future/label window
label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(COALESCE(gsc_impressions, 0)) AS imp_mar,
    SUM(COALESCE(gsc_clicks, 0)) AS clk_mar,
    COUNT(*) FILTER (WHERE gsc_data_available = TRUE) AS measured_mar_days
FROM {MAR}
GROUP BY client_hash_id, content_hash_id
""").df()

# Create label only from future information
frame = features.merge(
    label,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Keep rows with measured March data
frame = frame[frame["measured_mar_days"] > 0].copy()

frame["went_dark"] = (frame["clk_mar"] == 0).astype(int)

# Check for suspicious feature names
bad_words = [
    "mar", "future", "label", "outcome",
    "went_dark", "product", "optimized"
]

leak_features = [
    c for c in X.columns
    if any(word in c.lower() for word in bad_words)
]

print("Possible leakage features:", leak_features)

# Confirm future/label columns are NOT in X
for col in ["imp_mar", "clk_mar", "went_dark"]:
    print(col, "in X:", col in X.columns)

assert len(leak_features) == 0
assert "imp_mar" not in X.columns
assert "clk_mar" not in X.columns
assert "went_dark" not in X.columns

print("\nLeakage check passed.")

Possible leakage features: []
imp_mar in X: False
clk_mar in X: False
went_dark in X: False

Leakage check passed.


## 4. What I excluded and why

client_hash_id: excluded because it is an identifier used for joining, not a predictive feature.

content_hash_id: excluded because it is an identifier used for joining, not a predictive feature.

imp_mar: excluded because it comes from the future March label window.

clk_mar: excluded because it comes from the future March label window.

went_dark: excluded because it is the target label.

client_has_gsc: excluded because it is a product/data capability flag rather than content performance.

client_has_ga4: excluded because it is a product/data capability flag rather than content performance.

report_date: excluded because the prediction uses the fixed February feature window rather than the individual daily date.

In [20]:

excluded = {
    "client_hash_id": "identifier",
    "content_hash_id": "identifier",
    "imp_mar": "future information",
    "clk_mar": "future information",
    "went_dark": "target label",
    "client_has_gsc": "product/data capability flag",
    "client_has_ga4": "product/data capability flag",
    "report_date": "daily date not used as a feature"
}

print("Excluded fields:")
for field, reason in excluded.items():
    print(f"- {field}: {reason}")

print("\nFinal feature columns:")
print(X.columns.tolist())

Excluded fields:
- client_hash_id: identifier
- content_hash_id: identifier
- imp_mar: future information
- clk_mar: future information
- went_dark: target label
- client_has_gsc: product/data capability flag
- client_has_ga4: product/data capability flag
- report_date: daily date not used as a feature

Final feature columns:
['imp_feb', 'clk_feb', 'sessions_ai_feb', 'ga4_pageviews_feb', 'ga4_sessions_feb', 'ga4_engagement_sec_feb', 'ga4_available_feb', 'ctr_feb', 'avg_position_feb']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
-     [] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.